In [1]:
import sys
sys.path.append('../')
import time

import numpy as np
import matplotlib.pyplot as plt

import epics
from siriuspy.devices import CAXCtrl
from caxscripts.h5file import HDF5File

In [ ]:
CAX = CAXCtrl()

# Functions

Useful functions for the scan

In [ ]:
MAXERRORCOUNT = 5

In [ ]:

#todo: move to initial position

def current_parameters():
    return {
        'slit1': {
            'top': CAX.slit_A1.top_pos,
            'bottom': CAX.slit_A1.bottom_pos,
            'left': CAX.slit_A1.left_pos,
            'right': CAX.slit_A1.right_pos
        },
        'dvf1': {
            'expo_time': CAX.dvf_A1.acquisition_time
        },
        'dvf2': {
            'expo_time': CAX.dvf_A1.acquisition_time,
            'z_pos': CAX.dvf_B1.z_pos
        },
        'mirror': {
            'ry': CAX.mirror.ry_pos,
            'tx': CAX.mirror.tx_pos,
            'y1': CAX.mirror.y1_pos,
            'y2': CAX.mirror.y2_pos,
            'y3': CAX.mirror.y3_pos,
            'photocollector': CAX.mirror.photocurrent_signal
        }
    }

def move_all_slits(top, bottom, left, right):
    CAX.slit_A1.move_top(value=top)
    CAX.slit_A1.move_bottom(value=bottom)
    CAX.slit_A1.move_left(value=left)
    CAX.slit_A1.move_right(value=right)

def open_all_slits():
    move_all_slits(top=19.7-0.04,
                   bottom=35.8,
                   left=43.6-0.04,
                   right=47.2)

def move_robust_all_slits(top, bottom, left, right):
    CAX.slit_A1.move_robust_top(value=top)
    CAX.slit_A1.move_robust_bottom(value=bottom)
    CAX.slit_A1.move_robust_left(value=left)
    CAX.slit_A1.move_robust_right(value=right)

def get_image(dvf):

    count = 0
    while count < MAXERRORCOUNT:
        try:
            if not dvf.acquisition_status:
                dvf.cmd_acquire_on()
            return dvf.image
        except Exception as err:
            print(f" WARNING. When trying to fetch image from DVF1: {err} ")
            time.sleep(2)
            count += 1
            if count < MAXERRORCOUNT:
                print("\n Repeating the procedure...\n")
            else:
                raise Exception("Client exception")



# Scan

## Parameters

In [45]:
top_pos_open = 19.7 - 0.04
bottom_pos_open = 35.8
left_pos_open = 43.6 - 0.04
right_pos_open = 47.2

rangex = 1.3 + 0.1
rangey = 4.8

proport = rangey/rangex

In [5]:
L = 0.4

Ndxs = 7
Ndys = int(proport*Ndxs)
dxs = np.linspace(0,rangex-L,Ndxs)
dys = np.linspace(0,rangey-L,Ndys)

In [6]:
print('N of points:',Ndys*Ndxs)
# print(f'scan time estimative: {Ndys*Ndxs*4/60:.3f} min')

N of points: 161


# Execution

Initial state

In [27]:
local_time = time.localtime()
formatted_time = time.strftime("%Y%m%d-%H%M%S", local_time)
formatted_date = time.strftime("%Y%m%d", local_time)

formatted_time, formatted_date

('20250829-094858', '20250829')

In [ ]:
ANALYSIS = 'slit'

In [ ]:
parameters0 = current_parameters()

states_dir = '../states/'
state_name = f'{formatted_time}-{ANALYSIS}.txt'

with open(states_dir+state_name, 'w') as f:
    f.write(str(parameters0))

print(parameters0)

Initializate file

In [ ]:
scaname = f'scan_ry_{formatted_date}.h5'
datadir  = '/home/ids/data/'
direc    = f'{formatted_date}-Mirror-Ry/'

In [ ]:
file = HDF5File(filename=scaname, filedir=datadir+direc)

file.save_metadata(metadata_dict={
    'slit_top': CAX.slit_A1.top_pos,
    'slit_bottom': CAX.slit_A1.bottom_pos,
    'slit_left': CAX.slit_A1.left_pos,
    'slit_right': CAX.slit_A1.right_pos
})

Loop

In [ ]:
t0 = time.time()

for i, dy in enumerate(dys):

    move_robust_all_slits(top    = top_pos_open-rangey+L+dy,
                          bottom = bottom_pos_open-dy,
                          left   = left_pos_open,
                          right  = right_pos_open-rangex+L)

    for j, dx in enumerate(dxs):
        print(f'scan step x {j+1}/{len(dxs)} and step y {i+1}/{len(dys)}')

        CAX.slit_A1.move_robust_left(value=left_pos_open-dx)
        CAX.slit_A1.move_robust_right(value=right_pos_open-rangex+L+dx)

        img1 = get_image(CAX.dvf_A1)
        expotime1 = CAX.dvf_A1.exposure_time
        img2 = get_image(CAX.dvf_B1)
        expotime2 = CAX.dvf_B1.exposure_time

        scan = f'scan-{i:03d}-{j:03d}'
        scanmetadata = {
            'slit_top'    : CAX.slit_A1.top_pos,
            'slit_bottom' : CAX.slit_A1.bottom_pos,
            'slit_left'   : CAX.slit_A1.left_pos,
            'slit_right'  : CAX.slit_A1.right_pos
        }
        file.save_group(grpname=scan, grpmetadata=scanmetadata)
        file.save_dataset(grpname=scan, dsetname=f'dvf1', dsetmetadata={'expo_time':expotime1}, dsetdata=img1)
        file.save_dataset(grpname=scan, dsetname=f'dvf2', dsetmetadata={'expo_time':expotime2}, dsetdata=img2)


t1 = time.time()

print()
print(f'ellapsed time [s]: {t1-t0}')

move_all_slits(top    = top0,
               bottom = bottom0,
               left   = left0,
               right  = right0)

Done!
Done!
Done!is Moving... | New pos: 43.56 | Curr: 43.56 | Dif: 0.0
Done!is Moving... | New pos: 46.2 | Curr: 46.20 | Dif: 0.0
scan step x 1/7 and step y 1/23
Done!
Done!
scan step x 2/7 and step y 1/23
Done!is Moving... | New pos: 43.39333333333334 | Curr: 43.39 | Dif: 2.6041666664866625e-05
Done!is Moving... | New pos: 46.36666666666667 | Curr: 46.37 | Dif: 2.6041666664866625e-05
scan step x 3/7 and step y 1/23
Done!is Moving... | New pos: 43.22666666666667 | Curr: 43.23 | Dif: 2.6041666664866625e-05
Done!is Moving... | New pos: 46.53333333333334 | Curr: 46.53 | Dif: 2.6041666664866625e-05
scan step x 4/7 and step y 1/23
Done!is Moving... | New pos: 43.06 | Curr: 43.06 | Dif: 0.0
Done!is Moving... | New pos: 46.7 | Curr: 46.70 | Dif: 0.0
scan step x 5/7 and step y 1/23
Done!is Moving... | New pos: 42.89333333333334 | Curr: 42.89 | Dif: 2.6041666664866625e-05
Done!is Moving... | New pos: 46.86666666666667 | Curr: 46.87 | Dif: 2.6041666664866625e-05
scan step x 6/7 and step y 1/23


CA.Client.Exception...............................................
    Context: "10.30.14.19:33461"
    Source File: ../cac.cpp line 1237
    Current Time: Fri Aug 29 2025 10:28:14.814806222
..................................................................


Done!is Moving... | New pos: 46.53333333333334 | Curr: 46.53 | Dif: 2.6041666664866625e-05
 WARNING. When trying to fetch image from DVF1: 'NoneType' object cannot be interpreted as an integer 

 Repeating the procedure...

scan step x 4/7 and step y 3/23
Done!is Moving... | New pos: 43.06 | Curr: 43.06 | Dif: 0.0
Done!is Moving... | New pos: 46.7 | Curr: 46.70 | Dif: 0.0
scan step x 5/7 and step y 3/23
Done!is Moving... | New pos: 42.89333333333334 | Curr: 42.89 | Dif: 2.6041666664866625e-05
Done!is Moving... | New pos: 46.86666666666667 | Curr: 46.87 | Dif: 2.6041666664866625e-05
scan step x 6/7 and step y 3/23
Done!is Moving... | New pos: 42.72666666666667 | Curr: 42.73 | Dif: 2.6041666664866625e-05
Done!is Moving... | New pos: 47.03333333333334 | Curr: 47.03 | Dif: 2.6041666664866625e-05
scan step x 7/7 and step y 3/23
Done!is Moving... | New pos: 42.56 | Curr: 42.56 | Dif: 0.0
Done!is Moving... | New pos: 47.2 | Curr: 47.20 | Dif: 0.0
Done!is Moving... | New pos: 15.86 | Curr: 15.